# GPU transcription run (Kaggle or Google Colab)

* **Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On**, then *Save Version → Save & Run All*. The run continues in the background for up to 12 h; results stay in the *Output* tab. To resume in a new version, add the previous output as input: finished episodes are copied back and skipped.
* **Colab:** Runtime → Change runtime type → **T4 GPU** → Run all. Results are written straight to Google Drive, so after a disconnect just run again.
* **Several machines at once:** give each one different `MY_SHARDS` with the same `SHARD_COUNT`, and the same `SYNC_REMOTE` (any rclone remote). Each machine does its own shards first, then helps with the rest from the other end of the list.

In [ ]:
# === SETTINGS ===
FEED_URL = ""            # podcast RSS feed (or upload episodes.json and leave this empty)
SINCE = ""               # optional YYYY-MM-DD
MY_SHARDS = ""           # e.g. "0,1" on Kaggle and "2" on Colab with SHARD_COUNT = 3; "" = everything
SHARD_COUNT = 1
MODEL = "large-v3"
BATCH = 16               # T4 16 GB: 16; lower to 8 on out-of-memory errors
WORKERS_PER_GPU = 1
MAX_HOURS = 11.3         # stop taking new episodes before the platform's session limit
VOCABULARY = ""          # names and terms that appear in the show (initial prompt)
SYNC_REMOTE = ""         # optional rclone remote, e.g. "gdrive:podcast-archive"; config from secret RCLONE_CONF
REPO = "https://github.com/LektorGW/transcription-pipeline"

import os, sys, json, glob, shutil, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = not ON_KAGGLE and "google.colab" in sys.modules
if ON_KAGGLE:
    WORK = "/kaggle/working"
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/transcription-pipeline"
else:
    WORK = os.path.abspath("run")
os.makedirs(WORK, exist_ok=True)
print("working directory:", WORK)

In [ ]:
!git clone -q --depth 1 {REPO} /tmp/tp || git -C /tmp/tp pull -q
!pip -q install -e "/tmp/tp[local,xlsx]"
!pip -q install "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*" 2>/dev/null || true
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# === CONFIG + OPTIONAL SYNC ===
cfg = {"output_dir": f"{WORK}/output", "state_dir": "/tmp/tp_state", "vocabulary": VOCABULARY,
       "local": {"model": MODEL, "batch_size": BATCH, "device": "cuda", "compute_type": "float16"},
       "sync": {"remote": SYNC_REMOTE}}
json.dump(cfg, open("/tmp/tp_config.json", "w"))

conf = os.environ.get("RCLONE_CONF")
if SYNC_REMOTE and not conf and ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
    except Exception as ex:
        print("no RCLONE_CONF secret:", ex)
if SYNC_REMOTE and conf:
    if not shutil.which("rclone"):
        !curl -sO https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/
    os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
    open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w").write(conf)

# resume on Kaggle: copy transcripts from a previous version's output attached as input
for src in glob.glob("/kaggle/input/**/transcripts/*.json", recursive=True):
    dst = f"{WORK}/output/transcripts"
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(src[:-5] + ".*"):
        if not os.path.exists(os.path.join(dst, os.path.basename(f))):
            shutil.copy(f, dst)

In [ ]:
# === EPISODE LIST ===
if FEED_URL:
    !python -m transcription_pipeline episodes --rss "{FEED_URL}" --since "{SINCE}" -o {WORK}/episodes.json
eps = json.load(open(f"{WORK}/episodes.json"))
done = len(glob.glob(f"{WORK}/output/transcripts/*.json"))
print(f"episodes: {len(eps)} | already transcribed: {done}")

In [ ]:
# === RUN: N workers per GPU, progress every 5 minutes ===
!python -m transcription_pipeline launch --config /tmp/tp_config.json --episodes {WORK}/episodes.json \
    --workers-per-gpu {WORKERS_PER_GPU} --max-hours {MAX_HOURS} --shards "{MY_SHARDS}" --shard-count {SHARD_COUNT}

In [ ]:
# === SUMMARY + SEARCHABLE SITE ===
!python -m transcription_pipeline report --config /tmp/tp_config.json
!python -m transcription_pipeline site --config /tmp/tp_config.json -o {WORK}/site